In [2]:
from pathlib import Path

import geopandas as gpd
import networkx as nx
import osmnx as ox
import polars as pl

In [3]:
ROOT_PATH = Path(".").resolve().absolute()
DATASET_PATH = (
        ROOT_PATH / "sumo/simulations/ohare-chicago-junctionless/output/fcd.parquet"
)
NETWORK_PATH = ROOT_PATH / "networks/graphml/ohare_network.graphml"

In [4]:
G = ox.load_graphml(NETWORK_PATH)
edges_gdf = ox.graph_to_gdfs(G, nodes=False, fill_edge_geometry=True).to_crs(epsg=4326)
edges_gdf

,,,osmid,highway,lanes,name,oneway,ref,reversed,length,geometry,maxspeed,bridge
u,v,key,,,,,,,,,,,
1153867776,29839280,0,985432262,primary,3,Mannheim Road,True,US 12;US 45,False,22.870702,"LINESTRING (-87.87914 41.9887, -87.87925 41.98...",NaN,NaN
29839280,1153867817,0,985432262,primary,3,Mannheim Road,True,US 12;US 45,False,38.594048,"LINESTRING (-87.87925 41.98889, -87.87943 41.9...",NaN,NaN
1153867781,1153867758,0,11537207,motorway_link,1,NaN,True,NaN,False,30.909291,"LINESTRING (-87.88111 41.98211, -87.88126 41.9...",NaN,NaN
1153867758,102856723,0,11537207,motorway_link,1,NaN,True,NaN,False,19.860727,"LINESTRING (-87.88126 41.98185, -87.88133 41.9...",NaN,NaN
29786118,4686227143,0,320340442,tertiary,3,Bessie Coleman Drive,True,NaN,False,95.216379,"LINESTRING (-87.88573 41.97871, -87.88573 41.9...",30 mph,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29785981,5037315526,0,31296557,unclassified,2,O'Hare International Terminal Departures,True,NaN,False,8.552609,"LINESTRING (-87.89083 41.97663, -87.89092 41.9...",20 mph,NaN
1153867719,102857949,0,11537330,motorway_link,NaN,NaN,True,NaN,False,15.968346,"LINESTRING (-87.87667 41.97976, -87.87658 41.9...",NaN,NaN
1153867766,102855117,0,19012965,motorway_link,1,NaN,True,NaN,False,22.372636,"LINESTRING (-87.88105 41.98147, -87.88105 41.9...",NaN,NaN


In [5]:
lf = pl.scan_parquet(DATASET_PATH)
lf.collect().to_pandas()

,vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
0,0,"[-87.8856713296918, 41.99494072305388]",0.5,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885671,41.994941
1,0,"[-87.88567733180858, 41.994936271618215]",1.0,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885677,41.994936
2,0,"[-87.88568911243644, 41.994927534580114]",1.5,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885689,41.994928
3,0,"[-87.88570568973026, 41.99491498626672]",2.0,:263785883_1_0,:263785883_1_0,321574768,False,node_263785883,-87.885706,41.994915
4,0,"[-87.88572492532072, 41.994900100759565]",2.5,:263785883_1_0,:263785883_1_0,321574768,False,node_263785883,-87.885725,41.994900
...,...,...,...,...,...,...,...,...,...,...
859435,1702,"[-87.88874921463292, 41.99804074104869]",4081.5,:10021303714_0_0,:10021303714_0_0,1094214001,False,node_10021303714,-87.888749,41.998041
859436,1702,"[-87.88875558331904, 41.99798545992872]",4082.0,1094214001_0,1094214001_0,1094214001,False,1094214001,-87.888756,41.997985
859437,1702,"[-87.88875619867935, 41.997933337316766]",4082.5,1094214001_0,1094214001_0,1094214001,False,1094214001,-87.888756,41.997933
859438,1702,"[-87.88875677920512, 41.99788060455132]",4083.0,1094214001_0,1094214001_0,1094214001,False,1094214001,-87.888757,41.997881


In [6]:
from functools import lru_cache


@lru_cache(maxsize=None)
def find_pathway(
        G_road: nx.MultiDiGraph, source: int, target: int
) -> list[int] | None:
    try:
        path = nx.shortest_path(G_road, source=source, target=target, weight="length")
        edges_path = list((x, y, 0) for x, y in zip(path[:-1], path[1:]))
        edge_data = list(G.edges[edge]["osmid"] for edge in edges_path)

        res = []
        for ed in edge_data:
            if ed not in res:
                res.append(ed)
        return res

    except nx.NetworkXNoPath:
        return None

In [7]:
indexed_osm_edges = edges_gdf.reset_index().set_index("osmid")
indexed_osm_edges = indexed_osm_edges[["u", "v"]]

indexed_osm_edges = pl.from_pandas(indexed_osm_edges, include_index=True)

indexed_osm_edges = indexed_osm_edges.group_by("osmid").agg(
    pl.concat_list(pl.col("u"), pl.col("v")).alias("edges")
)
indexed_osm_edges

osmid,edges
i64,list[list[i64]]
868656149,"[[12085592676, 4686227139], [5397725896, 12542628565], … [8096883613, 8096883614]]"
1207735016,"[[310333143, 310333144], [310333144, 310333180], [310333180, 310333211]]"
1094426120,"[[12085592685, 4686227152]]"
28257292,"[[111028517, 111027765], [111027765, 1085575678], … [9288826171, 111027767]]"
1094128689,"[[263775920, 5037315488], [5037315488, 5037315489], … [5037315489, 263775921]]"
…,…
1094128683,"[[10020900037, 29785975]]"
1425945034,"[[310333155, 12793948402], [12793948402, 12793948405]]"
485427211,"[[4686227120, 4781950502], [4781950502, 11037327135], … [4781950504, 4781950503]]"


In [8]:
lf.collect()

vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
i64,"array[f64, 2]",f64,cat,cat,cat,bool,cat,f64,f64
0,"[-87.885671, 41.994941]",0.5,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885671,41.994941
0,"[-87.885677, 41.994936]",1.0,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885677,41.994936
0,"[-87.885689, 41.994928]",1.5,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885689,41.994928
0,"[-87.885706, 41.994915]",2.0,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885706,41.994915
0,"[-87.885725, 41.9949]",2.5,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885725,41.9949
…,…,…,…,…,…,…,…,…,…
1702,"[-87.888749, 41.998041]",4081.5,""":10021303714_0_0""",""":10021303714_0_0""","""1094214001""",false,"""node_10021303714""",-87.888749,41.998041
1702,"[-87.888756, 41.997985]",4082.0,"""1094214001_0""","""1094214001_0""","""1094214001""",false,"""1094214001""",-87.888756,41.997985
1702,"[-87.888756, 41.997933]",4082.5,"""1094214001_0""","""1094214001_0""","""1094214001""",false,"""1094214001""",-87.888756,41.997933


In [9]:
trajectories = lf

node_filtered_trajectories = trajectories.filter(
    pl.col("node_mapped_id").cat.starts_with("node_")
)

# clean step
node_filtered_trajectories = node_filtered_trajectories.with_columns(
    pl.col("edge_id")
    .cast(pl.String)
    .cast(pl.Int64, strict=False)
    .alias("next_edge_id"),
    pl.col("node_mapped_id")
    .cast(pl.String)
    .str.replace("node_", "")
    .cast(pl.Int64, strict=False)
    .alias("node_osmid"),
)

cleaned_node_filtered_trajectories = node_filtered_trajectories.drop_nulls(
    ["next_edge_id", "node_osmid"]
)

# Creates pairs of [next_edge_id, node_osmid] to check for connectivity
# pairs each node_osmid, which should be an edge_id, with the edge_id (which is being treated as the next_edge_id in the sequence, set in previous algorithm)
paired_filtered_trajectories = cleaned_node_filtered_trajectories.with_columns(
    pl.concat_list(
        pl.col("next_edge_id"),
        pl.col("node_osmid"),
    ).alias("pairs")
)

unique_pairs = paired_filtered_trajectories.select("pairs").unique()

unique_pairs = unique_pairs.with_columns(
    pl.col("pairs").list.get(0).alias("next_edge_id"),
)

# join with edges to get the u,v nodes of the edge to check for connectivity status
pairs_w_edges = unique_pairs.join(
    indexed_osm_edges.lazy(), left_on="next_edge_id", right_on="osmid", how="left"
)

pairs_w_edges = pairs_w_edges.with_columns(
    pl.col("pairs").list.get(0).alias("next_edge_id"),
    pl.col("pairs").list.get(1).alias("node_osmid"),
)
# Explodes the dataframe so each node of an edge gets its own row
exploded_pairs = pairs_w_edges.explode("edges")
exploded_pairs = exploded_pairs.with_columns(
    pl.col("edges").list.contains(pl.col("node_osmid")).alias("connected")
    # it means one of the nodes of the edge is the node_osmid
)

# Groups back by pair and checks if *any* of the edge's nodes matched
# aggregate to get whether there is any connection with the node_osmid and one of the edges
pairs_w_connectivity_status = exploded_pairs.group_by("pairs").agg(
    pl.col("connected").any().alias("has_connection")
)

# re-join with the original data
pairs_w_edges = pairs_w_edges.join(pairs_w_connectivity_status, on="pairs", how="left")

# filter out the rows where there is no connection
disconnected_pairs = pairs_w_edges.filter(pl.col("has_connection").not_()).drop(
    "has_connection"
)

# for each disconnected node, get its direct neighbors from the graph G
disconnected_pairs_w_neighbors = disconnected_pairs.with_columns(
    pl.col("node_osmid")
    .map_elements(G.neighbors, return_dtype=pl.List(pl.Int64))
    .alias("node_neighbors")
)

# identifies the best target node to connect to
disconnected_pairs_w_target_node = disconnected_pairs_w_neighbors.with_columns(
    [
        pl.struct("edges", "node_neighbors")
        .map_elements(
            lambda x: next(
                (u for u, v in x["edges"] if u in x["node_neighbors"]), x["edges"][0][0]
            ),
            return_dtype=pl.Int64,
        )
        .alias("node_to_connect")
    ]
)

# try to find a connection pathway from the disconnected node to the target node
disconnected_pairs_w_pathway = disconnected_pairs_w_target_node.with_columns(
    pl.struct("node_osmid", "node_to_connect")
    .map_elements(
        lambda x: find_pathway(G, x["node_osmid"], x["node_to_connect"]),
        return_dtype=pl.List(pl.Int64),
    )
    .alias("connection_pathway")
)

# filter out the rows where there is no connection pathway
disconnected_pairs_w_pathway = disconnected_pairs_w_pathway.filter(
    (pl.col("connection_pathway").is_not_null())
    & (pl.col("connection_pathway").list.len() > 0)
)

disconnected_pairs_w_pathway = disconnected_pairs_w_pathway.select(
    "next_edge_id", "node_osmid", "connection_pathway"
)

disconnected_pairs_w_pathway.collect()

next_edge_id,node_osmid,connection_pathway
i64,i64,list[i64]
1031067818,12085592687,"[518291384, 518291373]"
435904639,2610439065,[485427212]
1094426129,2610439065,[485427212]
4694834,311482827,"[435904640, 435904638]"
4694834,6775773838,"[435507834, 435904638]"
…,…,…
4685905,311482827,[435904640]
849300204,12085592684,[849300207]
321574768,2610439068,[1188647984]


In [10]:
disconnected_trajectories_w_pathway = cleaned_node_filtered_trajectories.join(
    disconnected_pairs_w_pathway,
    on=["next_edge_id", "node_osmid"],
    how="inner",
)
disconnected_trajectories_w_pathway.collect()

vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat,next_edge_id,node_osmid,connection_pathway
i64,"array[f64, 2]",f64,cat,cat,cat,bool,cat,f64,f64,i64,i64,list[i64]
4,"[-87.884379, 41.995029]",40.5,""":311482827_0_0""",""":311482827_0_0""","""4685905""",false,"""node_311482827""",-87.884379,41.995029,4685905,311482827,[435904640]
4,"[-87.884418, 41.99501]",41.0,""":311482827_0_0""",""":311482827_0_0""","""4685905""",false,"""node_311482827""",-87.884418,41.99501,4685905,311482827,[435904640]
4,"[-87.884446, 41.994999]",41.5,""":311482827_0_0""",""":311482827_0_0""","""4685905""",false,"""node_311482827""",-87.884446,41.994999,4685905,311482827,[435904640]
6,"[-87.885874, 41.976701]",66.0,""":cluster_12085592677_120855926…","""node_12085592680""","""727145351""",false,"""node_12085592680""",-87.885874,41.976701,727145351,12085592680,"[518291394, 518291392]"
6,"[-87.885867, 41.976691]",66.5,""":cluster_12085592677_120855926…","""node_12085592680""","""727145351""",false,"""node_12085592680""",-87.885867,41.976691,727145351,12085592680,"[518291394, 518291392]"
…,…,…,…,…,…,…,…,…,…,…,…,…
1702,"[-87.884393, 41.994921]",3997.0,""":311482827_1_0""",""":311482827_1_0""","""4685905""",false,"""node_311482827""",-87.884393,41.994921,4685905,311482827,[435904640]
1702,"[-87.884395, 41.994936]",3997.5,""":311482827_1_0""",""":311482827_1_0""","""4685905""",false,"""node_311482827""",-87.884395,41.994936,4685905,311482827,[435904640]
1702,"[-87.884401, 41.994955]",3998.0,""":311482827_1_0""",""":311482827_1_0""","""4685905""",false,"""node_311482827""",-87.884401,41.994955,4685905,311482827,[435904640]


In [11]:
row_identifier = ["vehicle_id", "time", "next_edge_id", "node_osmid"]

newly_created_path_rows = (
    disconnected_trajectories_w_pathway
    # 1. Explode the list of edges into separate rows
    .explode("connection_pathway")
    .with_columns(
        # 2. Create an index (0, 1, 2...) for each step in the path
        pl.cum_count("connection_pathway")
        .over(row_identifier)
        .alias("path_step_index")
    )
    .with_columns(
        # 3. Use the index to create the incremental timestamp
        (pl.col("time") + (pl.col("path_step_index") * 0.01)).alias("new_time"),

        # 4. Rename 'connection_pathway' to 'edge_id' to match the original schema
        pl.col("connection_pathway").alias("new_edge_id"),
    )
    # 5. Select and rename columns to build the final, clean DataFrame of new rows
    .select(
        pl.col("vehicle_id"),
        pl.col("new_time").alias("time"),
        pl.col("new_edge_id").alias("edge_id"), # This is the corrected edge
        pl.col("geo_position"),
        pl.col("raw_lane_id"),
        pl.col("node_mapped_id"),   
        pl.col("mapped_lane_id"),
        pl.col("lat"),
        pl.col("lon"),
        pl.col("reversed")
    )
)
newly_created_path_rows.collect()

vehicle_id,time,edge_id,geo_position,raw_lane_id,node_mapped_id,mapped_lane_id,lat,lon,reversed
i64,f64,i64,"array[f64, 2]",cat,cat,cat,f64,f64,bool
4,40.51,435904640,"[-87.884379, 41.995029]",""":311482827_0_0""","""node_311482827""",""":311482827_0_0""",41.995029,-87.884379,false
4,41.01,435904640,"[-87.884418, 41.99501]",""":311482827_0_0""","""node_311482827""",""":311482827_0_0""",41.99501,-87.884418,false
4,41.51,435904640,"[-87.884446, 41.994999]",""":311482827_0_0""","""node_311482827""",""":311482827_0_0""",41.994999,-87.884446,false
6,66.01,518291394,"[-87.885874, 41.976701]",""":cluster_12085592677_120855926…","""node_12085592680""","""node_12085592680""",41.976701,-87.885874,false
6,66.02,518291392,"[-87.885874, 41.976701]",""":cluster_12085592677_120855926…","""node_12085592680""","""node_12085592680""",41.976701,-87.885874,false
…,…,…,…,…,…,…,…,…,…
1702,3997.01,435904640,"[-87.884393, 41.994921]",""":311482827_1_0""","""node_311482827""",""":311482827_1_0""",41.994921,-87.884393,false
1702,3997.51,435904640,"[-87.884395, 41.994936]",""":311482827_1_0""","""node_311482827""",""":311482827_1_0""",41.994936,-87.884395,false
1702,3998.01,435904640,"[-87.884401, 41.994955]",""":311482827_1_0""","""node_311482827""",""":311482827_1_0""",41.994955,-87.884401,false


In [12]:
# An anti join returns rows from the left frame that do NOT have a match in the right frame.
original = lf
original = original.with_columns(
    pl.col("edge_id").cast(pl.String).cast(pl.Int64, strict=False),
)
good = original.join(newly_created_path_rows, on=["vehicle_id", "node_mapped_id", "raw_lane_id"], how="anti")
good.collect()

vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
i64,"array[f64, 2]",f64,cat,cat,i64,bool,cat,f64,f64
0,"[-87.885671, 41.994941]",0.5,"""1188647984_0""","""1188647984_0""",1188647984,false,"""1188647984""",-87.885671,41.994941
0,"[-87.885677, 41.994936]",1.0,"""1188647984_0""","""1188647984_0""",1188647984,false,"""1188647984""",-87.885677,41.994936
0,"[-87.885689, 41.994928]",1.5,"""1188647984_0""","""1188647984_0""",1188647984,false,"""1188647984""",-87.885689,41.994928
0,"[-87.885706, 41.994915]",2.0,""":263785883_1_0""",""":263785883_1_0""",321574768,false,"""node_263785883""",-87.885706,41.994915
0,"[-87.885725, 41.9949]",2.5,""":263785883_1_0""",""":263785883_1_0""",321574768,false,"""node_263785883""",-87.885725,41.9949
…,…,…,…,…,…,…,…,…,…
1702,"[-87.888749, 41.998041]",4081.5,""":10021303714_0_0""",""":10021303714_0_0""",1094214001,false,"""node_10021303714""",-87.888749,41.998041
1702,"[-87.888756, 41.997985]",4082.0,"""1094214001_0""","""1094214001_0""",1094214001,false,"""1094214001""",-87.888756,41.997985
1702,"[-87.888756, 41.997933]",4082.5,"""1094214001_0""","""1094214001_0""",1094214001,false,"""1094214001""",-87.888756,41.997933


In [13]:
newly_created_path_rows = newly_created_path_rows.select(good.collect_schema().names())
# Concatenate the original good rows with the newly generated path rows
corrected_trajectories_df = pl.concat(
    [good, newly_created_path_rows]
)

# Sort the final DataFrame to ensure trajectory integrity
corrected_trajectories_df = corrected_trajectories_df.sort("vehicle_id", "time")

# Now, execute the full lazy plan
final_result = corrected_trajectories_df.with_columns(
    pl.col("edge_id").cast(pl.String).cast(pl.Categorical)
)
final_result = final_result.collect()
final_result 

vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
i64,"array[f64, 2]",f64,cat,cat,cat,bool,cat,f64,f64
0,"[-87.885671, 41.994941]",0.5,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885671,41.994941
0,"[-87.885677, 41.994936]",1.0,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885677,41.994936
0,"[-87.885689, 41.994928]",1.5,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885689,41.994928
0,"[-87.885706, 41.994915]",2.0,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885706,41.994915
0,"[-87.885725, 41.9949]",2.5,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885725,41.9949
…,…,…,…,…,…,…,…,…,…
1758,"[-87.882045, 41.984797]",3760.5,"""-4685903_0""","""-4685903_0""","""4685903""",true,"""4685903""",-87.882045,41.984797
1758,"[-87.881951, 41.984797]",3761.0,"""-4685903_0""","""-4685903_0""","""4685903""",true,"""4685903""",-87.881951,41.984797
1758,"[-87.881856, 41.984797]",3761.5,"""-4685903_0""","""-4685903_0""","""4685903""",true,"""4685903""",-87.881856,41.984797


In [75]:
final_result.write_parquet(DATASET_PATH.with_name("fcd_resolved_2.parquet"), compression="zstd")

In [76]:
import pandas as pd

indexed_osm_edges = edges_gdf.reset_index().set_index("osmid")
df = lf.collect()
dfs_to_add = []
for row in df.filter(pl.col("node_mapped_id").cat.starts_with("node_")).iter_rows(
        named=True
):
    next_id = row.get("edge_id")
    if next_id is None:
        print(f"Skipping row with missing edge_id: {row['raw_lane_id']}")
        continue

    next_id = int(next_id)

    node_mapped_id = row.get("node_mapped_id")
    if node_mapped_id is None:
        print(f"Skipping row with missing node_mapped_id: {row}")
        continue

    if isinstance(node_mapped_id, str) and node_mapped_id.startswith("node_"):
        node_mapped_id = int(node_mapped_id.replace("node_", ""))
    else:
        node_mapped_id = int(node_mapped_id)

    try:
        sel = indexed_osm_edges.loc[next_id, ["u", "v"]]  # type: ignore
        if isinstance(sel, (pd.Series,)):
            sel_df = sel.to_frame().T
        else:
            sel_df = sel
        edge = sel_df.values.tolist()
        if len(edge) == 0:
            print(f"No edge found for node {next_id}")
            continue

        if node_mapped_id and all((node_mapped_id not in e) for e in edge):
            neighbors = G.neighbors(node_mapped_id)

            u, v = next(((u, v) for u, v in edge if u in neighbors), edge[0])

            connect_edges = find_pathway(G, node_mapped_id, u)
            if connect_edges:
                new_row = row.copy()
                del new_row["edge_id"]
                del new_row["time"]

                new_df = pl.DataFrame(
                    [
                        {
                            **new_row,
                            "edge_id": str(connect_edge),
                            "time": float(row["time"] + (i * 0.01)),
                        }
                        for i, connect_edge in enumerate(connect_edges)
                    ]
                )
                new_df = new_df.with_columns(
                    pl.col("time").cast(pl.Float64),
                    pl.col("raw_lane_id").cast(pl.Categorical),
                    pl.col("vehicle_id").cast(pl.Int64),
                    pl.col("geo_position").cast(pl.Array(pl.Float64, shape=2)),
                    pl.col("edge_id").cast(pl.Categorical),
                    pl.col("node_mapped_id").cast(pl.Categorical),
                    pl.col("mapped_lane_id").cast(pl.Categorical),
                )
                dfs_to_add.append(new_df)
                df = df.remove(pl.col("raw_lane_id") == row["raw_lane_id"])  # type: ignore

    except KeyError:
        print(f"KeyError: {node_mapped_id}")

Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :310333211_0_0
Skipping row with missing edge_id: :310333211_0_0
Skipping row with missing edge_id: :310333211_0_0
Skipping row with missing edge_id: :6378331511_0_0
Skipping row with missing edge_id: :311482791_1_0
Skipping row with missing edge_id: :311482791_1_0
Skipping row with missing edge_id: :311482791_1_0
Skipping row with missing edge_id: :311482827_1_0
Skipping row with missing edge_id: :311482827_1_0
Skipping row with missing edge_id: :311482827_1_0
Skipping row with missing edge_id: :11038570643_0_0
Skipping row with missing edge_id: :11038570643_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793878899_2_0
Skipping row with missing edge_

In [77]:
df_to_add = pl.DataFrame(
    {
        "time": pl.Int64,
        "vehicle_id": pl.Int64,
        "geo_position": pl.Array(pl.Float64, shape=2),
        "edge_id": pl.Categorical,
        "mapped_lane_id": pl.Categorical,
        "node_mapped_id": pl.Categorical,
        "raw_lane_id": pl.Categorical,
    }
)

if dfs_to_add:
    df_to_add = pl.concat(dfs_to_add)
    df_to_add = df_to_add.with_columns(
        pl.col("time").cast(pl.Float64),
        pl.col("vehicle_id").cast(pl.Int64),
        pl.col("geo_position").cast(pl.Array(pl.Float64, shape=2)),
        pl.col("edge_id").cast(pl.Categorical),
        pl.col("mapped_lane_id").cast(pl.Categorical),
        pl.col("node_mapped_id").cast(pl.Categorical),
        pl.col("raw_lane_id").cast(pl.Categorical),
    )

In [78]:
unique_edge_node_ids = df_to_add.select(pl.col("node_mapped_id", "edge_id")).unique()
unique_edge_node_ids = unique_edge_node_ids.with_columns(
    pl.col("edge_id").cast(pl.String).cast(pl.Int64),
    pl.col("node_mapped_id").cast(pl.String).str.replace("node_", "").cast(pl.Int64),
)
unique_edge_node_ids

node_mapped_id,edge_id
i64,i64
311482852,435904638
29786106,1043374338
310333172,435501496
12085592684,849300207
12085592684,849300204
…,…
29786106,1094426136
12085592677,518291392
310333172,4685746


In [79]:
df_to_add.describe()

statistic,vehicle_id,geo_position,raw_lane_id,mapped_lane_id,reversed,node_mapped_id,lon,lat,edge_id,time
str,f64,f64,str,str,f64,str,f64,f64,str,f64
"""count""",64275.0,64275.0,"""64275""","""64275""",64275.0,"""64275""",64275.0,64275.0,"""64275""",64275.0
"""null_count""",0.0,0.0,"""0""","""0""",0.0,"""0""",0.0,0.0,"""0""",0.0
"""mean""",889.477884,null,null,null,0.0,null,-87.887179,41.98402,null,2014.74026
"""std""",510.011112,null,null,null,null,null,0.004802,0.004523,null,1038.911976
"""min""",1.0,null,null,null,0.0,null,-87.905965,41.9765,null,22.0
"""25%""",461.0,null,null,null,null,null,-87.885642,41.980053,null,1156.05
"""50%""",879.0,null,null,null,null,null,-87.885582,41.984776,null,1990.01
"""75%""",1351.0,null,null,null,null,null,-87.885581,41.984859,null,2936.62
"""max""",1758.0,null,null,null,0.0,null,-87.879078,41.995177,null,3999.0


In [80]:
df_to_add = df_to_add.select(df.columns)

concatenated = pl.concat([df, df_to_add], how="vertical")
concatenated = concatenated.sort(["time", "vehicle_id"])
concatenated

vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
i64,"array[f64, 2]",f64,cat,cat,cat,bool,cat,f64,f64
0,"[-87.885671, 41.994941]",0.5,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885671,41.994941
0,"[-87.885677, 41.994936]",1.0,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885677,41.994936
0,"[-87.885689, 41.994928]",1.5,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885689,41.994928
0,"[-87.885706, 41.994915]",2.0,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885706,41.994915
0,"[-87.885725, 41.9949]",2.5,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885725,41.9949
…,…,…,…,…,…,…,…,…,…
1702,"[-87.888749, 41.998041]",4081.5,""":10021303714_0_0""",""":10021303714_0_0""","""1094214001""",false,"""node_10021303714""",-87.888749,41.998041
1702,"[-87.888756, 41.997985]",4082.0,"""1094214001_0""","""1094214001_0""","""1094214001""",false,"""1094214001""",-87.888756,41.997985
1702,"[-87.888756, 41.997933]",4082.5,"""1094214001_0""","""1094214001_0""","""1094214001""",false,"""1094214001""",-87.888756,41.997933


In [ ]:
concatenated.write_parquet(
    DATASET_PATH.with_name("fcd_resolved.parquet"), compression="zstd"
)